## Section 0 — Setup & Imports

In [ ]:
#  Standard library 
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

#  Add src/ to Python path 
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

#  Data manipulation 
import pandas as pd
import numpy as np

#  Prophet 
# Prophet expects a DataFrame with exactly two columns:
#   'ds' — the date column
#   'y'  — the target column (revenue in our case)
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

#  Visualisation 
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates

#  Shared utilities 
from utils.helpers import set_style, save_figure, start_logging
set_style()
log = start_logging(project_root, '03_02_prophet_baseline')

print('All imports successful')
print(f'Project root: {project_root}')

---
## Section 1 — Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'

# Load Prophet-format data (ds + y columns)
df_raw = pd.read_csv(processed_path / 'ts_prophet_weekly.csv',
                     parse_dates=['ds'])
df = df_raw.copy()

# Also load the full feature matrix for regime info
features_raw = pd.read_csv(processed_path / 'ts_features_weekly.csv',
                           parse_dates=['week_start'])
features = features_raw.copy()

print(f'Prophet dataset: {len(df):,} weeks')
print(f'Date range: {df["ds"].min().date()} → {df["ds"].max().date()}')
print(f'\nRevenue statistics:')
print(df['y'].describe().round(2))
print(f'\nZero revenue weeks: {(df["y"] == 0).sum()}')

In [ ]:
log(f'Prophet dataset: {len(df):,} weeks')
log(f'Date range: {df["ds"].min().date()} → {df["ds"].max().date()}')
log(f'\nRevenue statistics:')
log(df['y'].describe().round(2)) # pyright: ignore[reportArgumentType]
log(f'\nZero revenue weeks: {(df["y"] == 0).sum()}')

In [ ]:
#  Train / test split 
# Train: all data up to and including 2020-12-31
# Test:  January and February 2021 (the final weeks of the dataset)
#
# Why this split?
# We want to test on the most recent data — the period the model
# has never seen. If we tested on middle periods, the model could
# implicitly learn from future data during training.

CUTOFF_DATE = '2020-12-31'

train = df[df['ds'] <= CUTOFF_DATE].copy()
test  = df[df['ds'] >  CUTOFF_DATE].copy()

print(f'Training set: {len(train):,} weeks ({train["ds"].min().date()} → {train["ds"].max().date()})')
print(f'Test set:     {len(test):,} weeks  ({test["ds"].min().date()} → {test["ds"].max().date()})')
print(f'\nTest % of total: {len(test)/len(df)*100:.1f}%')

In [ ]:
log(f'Training set: {len(train):,} weeks ({train["ds"].min().date()} → {train["ds"].max().date()})')
log(f'Test set:     {len(test):,} weeks  ({test["ds"].min().date()} → {test["ds"].max().date()})')
log(f'\nTest % of total: {len(test)/len(df)*100:.1f}%')

## Section 2 — Fit the Prophet Model

In [ ]:
# Change-point identified by PELT in Notebook 1
CHANGE_POINT = '2020-02-24'

model = Prophet(
    changepoint_prior_scale = 0.3,
    seasonality_mode        = 'multiplicative',
    weekly_seasonality      = True, # pyright: ignore[reportArgumentType]
    yearly_seasonality      = True, # pyright: ignore[reportArgumentType]
    daily_seasonality       = False,  # weekly data — no daily pattern # pyright: ignore[reportArgumentType]
    changepoints            = [CHANGE_POINT]
)

# Fit on training data only
model.fit(train)

print('Prophet model fitted')
print(f'Training period: {train["ds"].min().date()} → {train["ds"].max().date()}')
print(f'Changepoint provided: {CHANGE_POINT}')
print(f'Seasonality mode: multiplicative')
print(f'Changepoint prior scale: 0.3')

In [ ]:
log('Prophet model fitted')
log(f'Training period: {train["ds"].min().date()} → {train["ds"].max().date()}')
log(f'Changepoint provided: {CHANGE_POINT}')
log(f'Seasonality mode: multiplicative')
log(f'Changepoint prior scale: 0.3')

## Section 3 — Generate Forecast

In [ ]:
# Create future DataFrame manually using the same Monday-anchored frequency
# as our source data — this avoids the date alignment mismatch
last_train_date = train['ds'].max()
test_dates      = pd.date_range(
    start=last_train_date + pd.Timedelta(weeks=1),
    periods=len(test) + 4,
    freq='W-MON'
)

future = pd.DataFrame({'ds': pd.concat([
    train['ds'],
    pd.Series(test_dates)
]).reset_index(drop=True)})

# Generate forecast
forecast = model.predict(future)

print(f'Forecast generated: {len(forecast):,} rows')
print(f'Forecast covers: {forecast["ds"].min().date()} → {forecast["ds"].max().date()}')
print()
print('Forecast columns available:')
print([c for c in forecast.columns if c in [
    'ds', 'yhat', 'yhat_lower', 'yhat_upper',
    'trend', 'weekly', 'yearly'
]])

# Verify dates align with test set
print()
print('Test dates in data:    ', test['ds'].dt.date.tolist())
print('Forecast dates sample: ', forecast['ds'].tail(12).dt.date.tolist())

In [ ]:
# Merge forecast with actual test values for evaluation
test_forecast = test.merge(
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']],
    on='ds', how='left'
)

# Clip negative forecasts to 0 — revenue cannot be negative
test_forecast['yhat'] = test_forecast['yhat'].clip(lower=0)

print('Test period forecast vs actuals:')
print(f'{"Week":<14} {"Actual":>12} {"Forecast":>12} {"Lower CI":>12} {"Upper CI":>12}')
print('-' * 64)
for _, row in test_forecast.iterrows():
    print(f'{str(row["ds"].date()):<14} '
          f'${row["y"]:>10,.0f} '
          f'${row["yhat"]:>10,.0f} '
          f'${row["yhat_lower"]:>10,.0f} '
          f'${row["yhat_upper"]:>10,.0f}')

## Section 4 — Model Evaluation

In [ ]:
def calculate_metrics(actual, predicted, label=''):
    """
    Calculate MAPE, MAE, and RMSE for a set of forecasts.
    Excludes weeks where actual = 0 from MAPE to avoid division by zero.
    """
    mask   = actual > 0  # exclude zero-revenue weeks from percentage calc
    actual_nz    = actual[mask]
    predicted_nz = predicted[mask]

    mape = np.mean(np.abs((actual_nz - predicted_nz) / actual_nz)) * 100
    mae  = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))

    print(f' {label} ' if label else ' Metrics ')
    print(f'  MAPE: {mape:.2f}%   (avg % error — lower is better)')
    print(f'  MAE:  ${mae:,.0f}   (avg dollar error per week)')
    print(f'  RMSE: ${rmse:,.0f}  (penalises large errors more)')
    print()
    return {'mape': mape, 'mae': mae, 'rmse': rmse}

prophet_metrics = calculate_metrics(
    test_forecast['y'],
    test_forecast['yhat'],
    label='Prophet — Test Set Performance'
)

# Also check training fit (in-sample)
train_forecast = train.merge(
    forecast[['ds', 'yhat']], on='ds', how='left'
)
train_forecast['yhat'] = train_forecast['yhat'].clip(lower=0)

train_metrics = calculate_metrics(
    train_forecast['y'],
    train_forecast['yhat'],
    label='Prophet — Training Set Performance (in-sample)'
)

print('Note: training MAPE will always be lower than test MAPE.')
print('A large gap between the two indicates overfitting.')

---
## Section 5 — Visualisations

In [ ]:
#  Chart 1: Full forecast vs actuals 
fig, ax = plt.subplots(figsize=(15, 6))

# Plot actual revenue
ax.plot(df['ds'], df['y'], color='#2E75B6', linewidth=1.5,
        label='Actual revenue', zorder=3)

# Plot Prophet forecast (full period)
ax.plot(forecast['ds'], forecast['yhat'], color='#ED7D31',
        linewidth=1.5, linestyle='--', label='Prophet forecast', zorder=2)

# Confidence interval
ax.fill_between(forecast['ds'],
                forecast['yhat_lower'].clip(lower=0),
                forecast['yhat_upper'],
                alpha=0.15, color='#ED7D31', label='95% confidence interval')

# Mark train/test boundary
ax.axvline(pd.to_datetime(CUTOFF_DATE), color='#C00000', # pyright: ignore[reportArgumentType]
           linestyle=':', linewidth=2, label=f'Train/test split ({CUTOFF_DATE})')

# Mark change-point
ax.axvline(pd.to_datetime(CHANGE_POINT), color='#70AD47', # pyright: ignore[reportArgumentType]
           linestyle=':', linewidth=1.5, label=f'COVID change-point ({CHANGE_POINT})')

ax.set_title('Prophet forecast vs actual weekly revenue',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Week')
ax.set_ylabel('Weekly revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')

save_figure(fig, '03_02_prophet_forecast_vs_actual.png')
plt.show()

In [ ]:
#  Chart 2: Test period close-up 
# Zoomed in on just the test period for detailed evaluation

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(test_forecast['ds'], test_forecast['y'],
        color='#2E75B6', linewidth=2, marker='o', markersize=6,
        label='Actual', zorder=3)
ax.plot(test_forecast['ds'], test_forecast['yhat'],
        color='#ED7D31', linewidth=2, marker='s', markersize=6,
        linestyle='--', label='Prophet forecast', zorder=2)
ax.fill_between(test_forecast['ds'],
                test_forecast['yhat_lower'].clip(lower=0),
                test_forecast['yhat_upper'],
                alpha=0.2, color='#ED7D31', label='95% CI')

# Annotate each point with the error
for _, row in test_forecast.iterrows():
    err = row['y'] - row['yhat']
    color = '#C00000' if err < 0 else '#1E6B3C'
    ax.annotate(f'${err:+,.0f}',
                xy=(row['ds'], max(row['y'], row['yhat'])),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color=color)

ax.set_title(f'Test period: Prophet forecast vs actual\n'
             f'MAPE: {prophet_metrics["mape"]:.1f}%   '
             f'MAE: ${prophet_metrics["mae"]:,.0f}/week',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Weekly revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')

save_figure(fig, '03_02_prophet_test_closeup.png')
plt.show()

In [ ]:
#  Chart 3: Prophet components decomposition 
# Prophet decomposes the forecast into trend + weekly + yearly seasonality
# This is one of Prophet's most useful features — it shows WHY
# the model made each prediction

fig = model.plot_components(forecast)
fig.suptitle('Prophet model components\n'
             '(Trend, weekly seasonality, yearly seasonality)',
             fontsize=12, fontweight='medium', y=1.01)
fig.set_size_inches(12, 8)

save_figure(fig, '03_02_prophet_components.png')
plt.show()

In [ ]:
#  Chart 4: Residuals analysis 
# Residuals = actual - predicted
# A well-calibrated model should have residuals that:
#   1. Are centred around zero (no systematic over/under prediction)
#   2. Have no obvious pattern over time (no autocorrelation in errors)
#   3. Are roughly normally distributed

train_forecast['residual'] = train_forecast['y'] - train_forecast['yhat']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Residuals over time
axes[0].plot(train_forecast['ds'], train_forecast['residual'],
             color='#2E75B6', linewidth=1, alpha=0.8)
axes[0].axhline(0, color='#C00000', linestyle='--', linewidth=1)
axes[0].axvline(pd.to_datetime(CHANGE_POINT), color='#70AD47',
                linestyle=':', linewidth=1.5, label=f'Change-point')
axes[0].set_title('Residuals over time\n(should be random around zero)',
                  fontsize=11, fontweight='medium')
axes[0].set_ylabel('Residual (actual − predicted)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].legend(fontsize=9)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Residual distribution
axes[1].hist(train_forecast['residual'], bins=25,
             color='#2E75B6', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='#C00000', linestyle='--', linewidth=1.5,
                label='Zero')
axes[1].axvline(train_forecast['residual'].mean(), color='#ED7D31',
                linestyle='--', linewidth=1.5,
                label=f'Mean: ${train_forecast["residual"].mean():,.0f}')
axes[1].set_title('Residual distribution\n(should be centred near zero)',
                  fontsize=11, fontweight='medium')
axes[1].set_xlabel('Residual (USD)')
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

fig.suptitle('Prophet residuals analysis', fontsize=13, fontweight='medium')
save_figure(fig, '03_02_prophet_residuals.png')
plt.show()

---
## Section 6 — Findings

In [ ]:
print('  NOTEBOOK 2 FINDINGS — PROPHET BASELINE MODEL')
print()
print('MODEL CONFIGURATION')
print(f'  Training period:          2019-01-07 → 2020-12-28')
print(f'  Test period:              2021-01-04 → 2021-02-22')
print(f'  Known change-point:       {CHANGE_POINT} (COVID lockdown)')
print(f'  Seasonality mode:         multiplicative')
print(f'  Changepoint prior scale:  0.3 (high flexibility)')
print()
print('PERFORMANCE METRICS — TEST SET')
print(f'  MAPE: {prophet_metrics["mape"]:.2f}%')
print(f'  MAE:  ${prophet_metrics["mae"]:,.0f} per week')
print(f'  RMSE: ${prophet_metrics["rmse"]:,.0f} per week')
print()
print('PERFORMANCE METRICS — TRAINING SET (in-sample)')
print(f'  MAPE: {train_metrics["mape"]:.2f}%')
print(f'  MAE:  ${train_metrics["mae"]:,.0f} per week')
print()
print('  Train/test MAPE gap: 1,037 percentage points')
print('  The model performed reasonably in training but failed')
print('  completely on the test set.')
print()
print('FINDING 1 — Test period revenue collapse is a data artefact')
print('  Actual revenue in Jan-Feb 2021 dropped from $41K to $3.6K')
print('  over 8 weeks — a near-total collapse. This is inconsistent')
print('  with the business trajectory and is almost certainly caused')
print('  by dataset truncation: orders placed in late 2020 with')
print('  future ship dates in 2021 were excluded from the Jan/Feb')
print('  counts due to the pre-order anomaly pattern found in')
print('  Project 01 (9.1% of orders had ship dates before purchase).')
print('  The test period does not represent real business performance.')
print()
print('FINDING 2 — Training performance is reasonable given constraints')
print('  Training MAPE of 25.13% with MAE of $5,728/week is')
print('  acceptable for a noisy 2-year dataset. The model correctly')
print('  learned the upward trend and the COVID regime shift.')
print()
print('FINDING 3 — Seasonality components show overfitting signs')
print('  Weekly seasonality swings of ±800% and yearly seasonality')
print('  of ±600% are unrealistically large. With only 2 years of')
print('  data, Prophet does not have enough cycles to reliably')
print('  learn true seasonal patterns — it is fitting to noise.')
print()
print('FINDING 4 — Dataset limitations are the root cause')
print('  This is the dataset constraint flagged before Project 03')
print('  began. A 2-year dataset with a mid-period structural break')
print('  (COVID) and a truncated endpoint gives Prophet too little')
print('  clean signal to produce reliable out-of-sample forecasts.')
print()
print('BENCHMARK FOR NOTEBOOK 3')
print(f'  Prophet training MAPE: {train_metrics["mape"]:.2f}% — used as benchmark')
print(f'  Test MAPE of 1,062% is not a meaningful benchmark due to')
print(f'  data truncation. LightGBM will be evaluated on training')
print(f'  MAPE and cross-validation rather than the collapsed test set.')
print()
print('Notebook 2 complete')
print('Figures saved     → reports/figures/  (4 charts)')
print(f'Training MAPE benchmark: {train_metrics["mape"]:.2f}%')


---
## Section 7 — Export Forecast

In [ ]:
# Save the full forecast including test actuals for model comparison
prophet_output = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper',
                            'trend', 'weekly', 'yearly']].copy()
prophet_output.columns = ['ds', 'prophet_yhat', 'prophet_lower',
                           'prophet_upper', 'prophet_trend',
                           'prophet_weekly', 'prophet_yearly']

# Merge with actuals
prophet_output = prophet_output.merge(df[['ds', 'y']], on='ds', how='left')
prophet_output = prophet_output.rename(columns={'y': 'actual_revenue'})

output_path = processed_path / 'prophet_forecast.csv'
prophet_output.to_csv(output_path, index=False)

print(f'Exported: prophet_forecast.csv')
print(f'Rows:     {len(prophet_output):,}')
print(f'Columns:  {list(prophet_output.columns)}')
print()

# Save metrics for comparison in Notebook 4
metrics_df = pd.DataFrame([{
    'model':       'Prophet',
    'mape':        round(prophet_metrics['mape'], 4),
    'mae':         round(prophet_metrics['mae'], 2),
    'rmse':        round(prophet_metrics['rmse'], 2),
    'train_mape':  round(train_metrics['mape'], 4),
    'test_weeks':  len(test),
    'train_weeks': len(train)
}])

metrics_path = processed_path / 'model_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'Exported: model_metrics.csv (benchmark for Notebook 4)')